# Object Tracking in Computer Vision

## 1. Introduction to Object Tracking

Object tracking is the process of:
1. Locating an object in successive frames of a video
2. Maintaining object identity across frames
3. Generating trajectory of the object over time

### Applications:
- Video surveillance
- Human-computer interaction
- Traffic monitoring
- Augmented reality
- Medical imaging

### Challenges:
- Occlusion
- Scale variations
- Illumination changes
- Motion blur
- Real-time requirements

## Single-Frame vs Multi-Frame Tracking

- **Single-Object Tracking (SOT)**: Tracks a single tagret across frames, typically initialized with a bounding box in first time

- **Multi-Object Tracking (MOT)**: Detects multiple objects per frame (via a detector like YOLOv8), then links detections across time to mantain unique ids. MOT is typically implemeneted with a tracking by detection strategy 

## 2. Tracking Algorithms Overview

### SORT (Simple Online and Realtime Tracking)
- **Key Features:**
  - Kalman Filter for motion prediction
  - Hungarian algorithm for data association
  - IOU-based matching
- **Pros:**
  - Fast and efficient
  - Simple implementation
  - Good for real-time applications
- **Cons:**
  - High number of ID switches
  - Poor handling of occlusions

### DeepSORT
- **Key Features:**
  - Deep appearance features
  - Mahalanobis distance matching
  - Cascade matching strategy
- **Pros:**
  - More robust in occlusions
  - Better ID preservation
  - Good balance of speed and accuracy
- **Cons:**
  - Requires GPU for optimal performance
  - More complex than SORT

### ByteTrack
- **Key Features:**
  - BYTE association
  - Low score detection utilization
  - Two-stage association
- **Pros:**
  - State-of-the-art performance
  - Better handling of crowded scenes
  - Robust to occlusions
- **Cons:**
  - Most complex implementation
  - Higher computational requirements

## 3. Evaluation Metrics

In Multi-Object Tracking (MOT), we evaluate how well a model can:
- Detect objects  
- Maintain object identities over time

We focus on 3 popular metrics: **MOTA**, **IDF1**, and **HOTA**.

### Matching with Hungarian Algorithm

Used to match predicted boxes with ground truth (GT) per frame by maximizing **IoU**.

From the matching, we get:
- **TP (True Positives)** – correct matches  
- **FP (False Positives)** – unmatched predictions  
- **FN (False Negatives)** – unmatched ground truths  


### MOTA (Multiple Object Tracking Accuracy)

Combines:
- Detection errors: FP, FN  
- Identity errors: IDSW (identity switches)

**Formula:**

$$
\mathrm{MOTA} \;=\; 1 \;-\; \frac{\displaystyle\sum_{t}\bigl(\mathrm{FN}_t + \mathrm{FP}_t + \mathrm{IDSW}_t\bigr)}{\displaystyle\sum_{t}\mathrm{GT}_t}
$$

**Pros:**
- Simple and widely used

**Cons:**
- Sensitive to crowded scenes  
- Ignores long-term ID consistency


### IDF1 (ID Precision and Recall F1-Score)

Focuses on **identity consistency** over the whole video.

**Formula:**

$$
\mathrm{IDF1} \;=\; \frac{2 \,\mathrm{IDTP}}{2\,\mathrm{IDTP} + \mathrm{IDFP} + \mathrm{IDFN}}
$$

Where:
- **IDTP**: Correctly identified detections  
- **IDFP**: Wrong identities  
- **IDFN**: Missed tracks

**Pros:**
- Good for long-term ID tracking

**Cons:**
- May oversimplify identity switches


### HOTA (Higher Order Tracking Accuracy)

Balances **detection** and **association** accuracy.

Breaks down into:
- **DetA** – detection accuracy  
- **AssA** – identity association accuracy

**Formula:**

$$
\mathrm{HOTA} \;=\; \sqrt{\mathrm{DetA} \;\times\; \mathrm{AssA}}
$$

Details:
- Uses multiple IoU thresholds (0.05 to 0.95)  
- Evaluates the entire video sequence

**Pros:**
- Balanced and robust

**Cons:**
- Not ideal for online tracking (uses future info)


| Metric | Focus                | Pros                              | Cons                              |
|--------|----------------------|-----------------------------------|-----------------------------------|
| MOTA   | Detection + ID Switch | Simple, interpretable             | Ignores long-term ID issues       |
| IDF1   | Identity Matching     | Focuses on tracking consistency   | Can ignore detection quality      |
| HOTA   | Detection + Association | Balanced, newer standard         | Needs full sequence (offline only)|


### When to Use What?

- **MOTA**: Quick overview, basic cases  
- **IDF1**: Important to track same object correctly  
- **HOTA**: Best for balanced evaluation (default choice today)

## 4. Kalman Filter Basics

The Kalman Filter is a recursive state estimator used in tracking algorithms.

### Components:

1. **State Vector**
   - Position (x, y)
   - Velocity (dx/dt, dy/dt)
   - Acceleration (optional)

2. **Prediction Step**
   - Predicts next state based on motion model
   - Updates uncertainty

3. **Update Step**
   - Incorporates new measurements
   - Updates state estimate and uncertainty

### Role in Tracking:
- Handles missing detections
- Smooths trajectories
- Predicts future positions